# Pulmonary airway segmentation

Segment airways in chest CT using the pretrained EXACT09-derived workflow.
**Inputs:** local scalar 3-D CT in Hounsfield units, at least 160 voxels per
axis. Set `ANTSTORCH_EXAMPLE_CHEST_CT` or the path cell below. An optional
binary region mask can be supplied with `ANTSTORCH_EXAMPLE_LUNG_MASK`.
**Resources:** downloads model weights and, if needed, lung-extraction priors
and weights. Patches are 160³; even batch size 1 can require substantial memory.
CPU inference can be slow. Do not normalize or convert HU values before inference.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import ants
import antstorch

# Display figures inside Jupyter/VS Code even if MPLBACKEND was set to Agg.
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

device = "cpu"
antstorch.set_default_device(device)

In [ ]:
import os
from pathlib import Path

ct_path = os.environ.get("ANTSTORCH_EXAMPLE_CHEST_CT")
mask_path = os.environ.get("ANTSTORCH_EXAMPLE_LUNG_MASK")  # Optional binary patch-selection mask.
# Alternatively set existing absolute paths here.
if ct_path is None or not Path(ct_path).is_file():
    raise FileNotFoundError("Set ct_path or ANTSTORCH_EXAMPLE_CHEST_CT to a scalar 3-D chest CT in HU.")
ct = ants.image_read(str(ct_path)).clone("float")
assert ct.dimension == 3 and not ct.has_components
assert np.isfinite(ct.numpy()).all() and np.ptp(ct.numpy()) > 0
if min(ct.shape) < 160:
    raise ValueError("The patch model requires at least 160 voxels along every image axis.")
lung_mask = None
if mask_path is not None:
    if not Path(mask_path).is_file():
        raise FileNotFoundError(f"Mask does not exist: {mask_path}")
    lung_mask = ants.image_read(str(mask_path)).clone("float")
    assert lung_mask.dimension == 3 and not lung_mask.has_components
    assert lung_mask.shape == ct.shape
    assert np.isfinite(lung_mask.numpy()).all()
    assert np.isin(lung_mask.numpy(), [0, 1]).all()
    assert (lung_mask.numpy() > 0).any()
    for attribute in ("spacing", "origin", "direction"):
        np.testing.assert_allclose(getattr(lung_mask, attribute), getattr(ct, attribute))
print("CT shape/spacing:", ct.shape, ct.spacing)

## Predict overlapping patches
Without a supplied mask, the function estimates lung labels and builds its
airway-inclusive, dilated patch-selection region. A tight lung-only mask can
omit central airways; inspect the region if you supply a custom mask.
The mask controls patch extraction, not a final thresholded segmentation.
Start with batch size 1 for lower peak prediction memory; larger batches trade
memory for throughput. The default stride of 32 retains overlapping coverage.

In [ ]:
probability = antstorch.lung_airway_segmentation(
    ct, lung_mask=lung_mask, prediction_batch_size=1,
    patch_stride_length=32, device=device, verbose=True,
)
values = probability.numpy()
assert probability.shape == ct.shape and np.isfinite(values).all()
assert values.min() >= -1e-6 and values.max() <= 1 + 1e-6
for attribute in ("spacing", "origin", "direction"):
    np.testing.assert_allclose(getattr(probability, attribute), getattr(ct, attribute))
threshold = 0.5  # Demonstration value; validate it for the intended acquisition.
mask = ants.threshold_image(probability, threshold, 1., 1, 0)
display_ct = ct.clone()
display_ct[display_ct < -1000] = -1000
display_ct[display_ct > 400] = 400
ants.plot(display_ct, overlay=mask, axis=2, nslices=8,
          overlay_cmap="autumn", overlay_alpha=0.6)
output_dir = Path("example_outputs/lung_airway_segmentation")
output_dir.mkdir(parents=True, exist_ok=True)
ants.image_write(probability, str(output_dir / "probability.nii.gz"))
ants.image_write(mask, str(output_dir / "mask.nii.gz"))
volume = float(np.count_nonzero(mask.numpy()) * np.prod(ct.spacing))
print(f"Thresholded volume: {volume:.1f} cubic physical units")
print("Outputs (matching files are overwritten):", output_dir.resolve())

Inspect continuity, small branches, false positives and image boundaries.
This notebook displays a binary mask at 0.5 while retaining the probability
image on disk. The threshold and model performance must be assessed on the
intended images; neither finite probabilities nor a nonempty mask establishes
anatomical accuracy. Correct physical geometry and CT units are essential.